# 第 3 章 線形回帰

部屋数から住宅価格を予測する線形回帰を、simple / absolute / square の 3 つのトリックで学習します。

対応する記事: [第 3 章 線形回帰（Jupyter Notebook（Python） の言語版）](../../../docs/article/grokking-machine-learning/python/ch03.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch03_linear_regression import *

## データセット

原著と同じ 6 件の住宅データです。特徴量は部屋数、ラベルは価格です。

In [2]:
features = [1.0, 2.0, 3.0, 5.0, 6.0, 7.0]
labels = [155.0, 197.0, 244.0, 356.0, 407.0, 448.0]

for x, y in zip(features, labels):
    print(f"部屋数 {x:.0f} → 価格 {y:.0f}")

部屋数 1 → 価格 155
部屋数 2 → 価格 197
部屋数 3 → 価格 244
部屋数 5 → 価格 356
部屋数 6 → 価格 407
部屋数 7 → 価格 448


## 3 つのトリックを 1 点分だけ試す

予測が `50 × 3 + 100 = 250`、正解が 300 のとき、それぞれのトリックがどう動くかを見ます。

二乗トリックだけが **誤差の大きさに比例** して動くことに注目してください。

In [3]:
model = Model(slope=50.0, intercept=100.0)
print("予測", model.predict(3.0), "正解 300.0")

print("absolute", absolute_trick(model, rooms=3.0, price=300.0, learning_rate=0.01))
print("square  ", square_trick(model, rooms=3.0, price=300.0, learning_rate=0.01))

予測 250.0 正解 300.0
absolute Model(slope=50.03, intercept=100.01)
square   Model(slope=51.5, intercept=100.5)


## 学習

学習率 0.01、1000 エポックで学習します。真の関係は「傾き 50・切片 100」です。

In [4]:
trained, errors = linear_regression(features, labels, learning_rate=0.01, epochs=1000, seed=0)

print(f"傾き   {trained.slope:.4f}")
print(f"切片   {trained.intercept:.4f}")
print(f"RMSE   {model_rmse(trained, features, labels):.4f}")
print(f"初期の RMSE {errors[0]:.2f} → 最終 {errors[-1]:.4f}")

傾き   51.0443
切片   91.5945
RMSE   7.4501
初期の RMSE 315.77 → 最終 7.1389


## 誤差の推移

エポックごとの RMSE を 100 エポック刻みで見ます。**最初の数百エポックで大きく下がり、その後は緩やかになります。**

In [5]:
for epoch in range(0, 1000, 100):
    bar = "#" * int(errors[epoch] / 8)
    print(f"epoch {epoch:4d}  RMSE {errors[epoch]:7.3f}  {bar}")

epoch    0  RMSE 315.767  #######################################
epoch  100  RMSE  33.402  ####
epoch  200  RMSE  26.398  ###
epoch  300  RMSE  21.650  ##
epoch  400  RMSE  16.591  ##
epoch  500  RMSE  13.236  #
epoch  600  RMSE  11.404  #
epoch  700  RMSE   9.346  #
epoch  800  RMSE   7.845  
epoch  900  RMSE   7.014  


## 予測してみる

学習したモデルで、部屋数 4 の家の価格を予測します。

In [6]:
for rooms in [1.0, 4.0, 8.0]:
    print(f"部屋数 {rooms:.0f} → 予測価格 {trained.predict(rooms):.2f}")

部屋数 1 → 予測価格 142.64
部屋数 4 → 予測価格 295.77
部屋数 8 → 予測価格 499.95


## 試してみる

学習率を変えると何が起きるでしょうか。**大きすぎると発散し、小さすぎると収束しません。**

In [7]:
for rate in [0.001, 0.01, 0.1]:
    m, e = linear_regression(features, labels, learning_rate=rate, epochs=1000, seed=0)
    print(f"学習率 {rate:<6} 傾き {m.slope:9.4f}  RMSE {model_rmse(m, features, labels):10.4f}")

学習率 0.001  傾き   63.0080  RMSE    32.9088
学習率 0.01   傾き   51.0443  RMSE     7.4501
学習率 0.1    傾き 1955966577352813873254062668257748706131968.0000  RMSE 12269022185894522106604965076303038356914176.0000
